<a href="https://colab.research.google.com/github/tatipara/eleicao_2026_1turno/blob/main/elei%C3%A7%C3%A3o2026_1turno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install requests pandas

In [21]:
# -*- coding: utf-8 -*-
"""
Votos para Presidente - Eleições 2026 (1º turno) - por município e por zona eleitoral
Saídas (em /content/tse2026/):
  - presidente_2026_municipios_long.csv  -> uma linha por município x candidato
  - presidente_2026_municipios_wide.csv  -> uma linha por município, uma coluna por candidato
  - presidente_2026_zonas_long.csv       -> uma linha por município x zona x candidato
  - presidente_2026_resumo_municipios.csv-> eleitorado, comparecimento, brancos, nulos etc.
  - raw/                                 -> JSON bruto de cada município (auditoria/reprocessamento)
"""

import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

# ----------------------------------------------------------------------------
# 1) PARÂMETROS
# ----------------------------------------------------------------------------
CICLO = "ele2026"
ELEICAO = "6257"        # 1º turno federal (Presidente). 2º turno: "6258"
CARGO = "0001"          # Presidente
INCLUIR_EXTERIOR = True # UF "zz" = cidades no exterior
UFS = None              # None = todas; ou lista, ex.: ["sp", "rj"]
THREADS = 8             # reduza se receber erros 403/429
SALVAR_BRUTO = True

BASE = f"https://resultados.tse.jus.br/oficial/{CICLO}/{ELEICAO}"
ELE6 = ELEICAO.zfill(6)
OUT = Path("/content/tse2026") if Path("/content").exists() else Path("tse2026")
RAW = OUT / "raw"
RAW.mkdir(parents=True, exist_ok=True)

sessao = requests.Session()
sessao.headers.update({"User-Agent": "Mozilla/5.0 (pesquisa academica; Colab)"})


# ----------------------------------------------------------------------------
# 2) FUNÇÕES AUXILIARES
# ----------------------------------------------------------------------------
def baixar_json(url, tentativas=4):
    """GET com novas tentativas e espera progressiva. Retorna dict ou None (404)."""
    for i in range(tentativas):
        try:
            r = sessao.get(url, timeout=30)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return json.loads(r.content.decode("utf-8-sig"))
        except Exception as e:  # noqa: BLE001
            if i == tentativas - 1:
                print(f"FALHA: {url} -> {e}")
                return None
            time.sleep(1.5 * (i + 1))


def num(x):
    """Converte número em formato pt-BR ('1.234' / '12,34') para int/float."""
    if not isinstance(x, str):
        return x
    s = x.strip()
    if s == "":
        return None
    try:
        if "," in s:
            return float(s.replace(".", "").replace(",", "."))
        return int(s.replace(".", ""))
    except ValueError:
        return x


def escalares(d, prefixo=""):
    """Extrai apenas os campos escalares de um dict (com prefixo no nome)."""
    return {f"{prefixo}{k}": num(v) for k, v in d.items()
            if not isinstance(v, (dict, list))}


def mapa_candidatos(no, partido=None, acc=None):
    """Percorre a árvore 'carg' e monta {numero: {nome, partido, sqcand}}."""
    acc = {} if acc is None else acc
    if isinstance(no, dict):
        partido = no.get("sg", partido) if "cand" in no else partido
        if "n" in no and ("nmu" in no or "sqcand" in no):
            acc[str(no["n"])] = {
                "candidato": no.get("nmu") or no.get("nm"),
                "nome_completo": no.get("nm"),
                "partido": partido,
                "sqcand": no.get("sqcand"),
            }
        for v in no.values():
            mapa_candidatos(v, partido, acc)
    elif isinstance(no, list):
        for v in no:
            mapa_candidatos(v, partido, acc)
    return acc


def processar_municipio(m):
    """Baixa o arquivo do município e devolve (linhas_mun, linhas_zona, resumo)."""
    uf, cd = m["uf"], m["cd_tse"]
    url = f"{BASE}/dados/{uf}/{uf}{cd}-c{CARGO}-e{ELE6}-u.json"
    j = baixar_json(url)
    if j is None:
        return [], [], None
    if SALVAR_BRUTO:
        (RAW / f"{uf}{cd}.json").write_text(json.dumps(j, ensure_ascii=False), encoding="utf-8")

    cands = mapa_candidatos(j.get("carg", []))
    ident = {"uf": uf.upper(), "cd_tse": cd, "cd_ibge": m["cd_ibge"], "municipio": m["nome"]}
    lin_mun, lin_zona, resumo = [], [], None

    for a in j.get("abr", []):
        tipo = str(a.get("tpabr", "")).upper()
        geral = escalares(a)
        for bloco in ("s", "e", "v"):          # seções, eleitorado, votos
            if isinstance(a.get(bloco), dict):
                geral.update(escalares(a[bloco], f"{bloco}_"))
        if tipo.startswith("MU"):
            resumo = {**ident, **geral}
        for c in a.get("cand", []):
            n = str(c.get("n"))
            linha = {**ident, "numero": n, **cands.get(n, {}), **escalares(c)}
            if tipo.startswith("MU"):
                lin_mun.append(linha)
            elif tipo.startswith("Z"):
                lin_zona.append({**linha, "zona": a.get("cdabr")})

    # Reserva: se o layout não trouxer 'abr', usa os votos dentro de 'carg'
    if not lin_mun:
        def varrer(no):
            if isinstance(no, dict):
                if "n" in no and "vap" in no:
                    n = str(no["n"])
                    lin_mun.append({**ident, "numero": n, **cands.get(n, {}), **escalares(no)})
                for v in no.values():
                    varrer(v)
            elif isinstance(no, list):
                for v in no:
                    varrer(v)
        varrer(j.get("carg", []))
    return lin_mun, lin_zona, resumo


# ----------------------------------------------------------------------------
# 3) CONFERE O CATÁLOGO DE ELEIÇÕES E LISTA OS MUNICÍPIOS
# ----------------------------------------------------------------------------
cat = baixar_json("https://resultados.tse.jus.br/oficial/comum/config/ele-c.json")
if cat is not None and ELEICAO not in json.dumps(cat):
    print(f"ATENÇÃO: o código {ELEICAO} não aparece em ele-c.json. Confira o catálogo.")

cfg = baixar_json(f"{BASE}/config/mun-e{ELE6}-cm.json")
assert cfg is not None, "Não foi possível baixar a lista de municípios (verifique BASE/ELEICAO)."

municipios = []
for uf in cfg["abr"]:
    sigla = uf["cd"].lower()
    if sigla == "zz" and not INCLUIR_EXTERIOR:
        continue
    if UFS and sigla not in [u.lower() for u in UFS]:
        continue
    for mu in uf.get("mu", []):
        municipios.append({
            "uf": sigla,
            "cd_tse": str(mu["cd"]).zfill(5),
            "cd_ibge": mu.get("cdi"),
            "nome": mu.get("nm"),
        })
print(f"{len(municipios)} localidades a baixar.")

# ----------------------------------------------------------------------------
# 4) DOWNLOAD PARALELO
# ----------------------------------------------------------------------------
L_MUN, L_ZONA, L_RES, falhas = [], [], [], []
with ThreadPoolExecutor(max_workers=THREADS) as ex:
    futuros = {ex.submit(processar_municipio, m): m for m in municipios}
    for f in tqdm(as_completed(futuros), total=len(futuros)):
        a, b, r = f.result()
        if not a:
            falhas.append(futuros[f])
        L_MUN += a
        L_ZONA += b
        if r:
            L_RES.append(r)

# ----------------------------------------------------------------------------
# 5) TABELAS E EXPORTAÇÃO
# ----------------------------------------------------------------------------
df = pd.DataFrame(L_MUN)
dz = pd.DataFrame(L_ZONA)
dr = pd.DataFrame(L_RES)

# 'vap' = votos apurados do candidato; 'pvap' = percentual sobre os votos válidos
df = df.rename(columns={"vap": "votos", "pvap": "pct_validos"})
dz = dz.rename(columns={"vap": "votos", "pvap": "pct_validos"})

df.to_csv(OUT / "presidente_2026_municipios_long.csv", index=False, encoding="utf-8-sig")
dz.to_csv(OUT / "presidente_2026_zonas_long.csv", index=False, encoding="utf-8-sig")
dr.to_csv(OUT / "presidente_2026_resumo_municipios.csv", index=False, encoding="utf-8-sig")

if {"votos", "candidato"}.issubset(df.columns):
    wide = (df.pivot_table(index=["uf", "cd_tse", "cd_ibge", "municipio"],
                           columns="candidato", values="votos", aggfunc="sum")
              .reset_index())
    wide.to_csv(OUT / "presidente_2026_municipios_wide.csv", index=False, encoding="utf-8-sig")

    print("\nTotal nacional reconstruído pela soma dos municípios (conferir com o TSE):")
    tot = df.groupby("candidato")["votos"].sum().sort_values(ascending=False)
    print(pd.DataFrame({"votos": tot, "pct": (100 * tot / tot.sum()).round(2)}))
else:
    print("Layout inesperado. Colunas encontradas:", list(df.columns))
    print("Inspecione um arquivo em", RAW, "e ajuste os nomes dos campos.")

print(f"\nMunicípios com dados: {df['cd_tse'].nunique() if len(df) else 0} | falhas: {len(falhas)}")
if falhas:
    pd.DataFrame(falhas).to_csv(OUT / "falhas.csv", index=False)
    print("Lista de falhas salva em falhas.csv (execute novamente para tentar de novo).")
print("Arquivos em:", OUT)

# Para baixar tudo do Colab em um único .zip, descomente:
# import shutil; from google.colab import files
# shutil.make_archive("/content/tse2026", "zip", OUT); files.download("/content/tse2026.zip")

5757 localidades a baixar.


  0%|          | 0/5757 [00:00<?, ?it/s]


Total nacional reconstruído pela soma dos municípios (conferir com o TSE):
                              votos    pct
candidato                                 
FLAVIO BOLSONARO           56104503  47.03
LULA                       53879538  45.16
ESCRITOR AUGUSTO CURY       3448569   2.89
RENAN SANTOS                2675887   2.24
RONALDO CAIADO              2605148   2.18
ZEMA                         326488   0.27
SAMARA                       122911   0.10
HERTZ DIAS                    43103   0.04
CLARIANA BARAO                40043   0.03
EDMILSON COSTA                22693   0.02
VETERINÁRIO WILSON GRASSI     16881   0.01
RUI COSTA PIMENTA             15024   0.01

Municípios com dados: 5757 | falhas: 0
Arquivos em: /content/tse2026
